### tensorium

tensorium was born from the need for a lightweight, high-performance tensor library that puts control and optimization back in the hands of researchers and developers — initially driven by a personal need to build a complete BSSN solver from scratch. Designed to be both modular and powerful, **tensorium** emphasizes aligned memory, SIMD acceleration (SSE, AVX2, AVX512), multithreading, and optional MPI support — all without sacrificing readability or flexibility.  

The motivation behind **tensorium** stems from a simple but demanding observation: many critical applications — ranging from simulations of spacetime geometry to machine learning training loops — require the repeated manipulation of large, dense vectors, matrices, and tensors. These operations must be performed as efficiently as possible on modern hardware, ideally without sacrificing clarity or interoperability.
s

Here is how to use the tensorium lib with python (search for the path because it was not used with pip and installed in the correct path):

In [35]:
import sys
import time 
import os
sys.path.append(os.path.abspath("./pysrc"))
from tensorium import *
from tensorium import Matrix, tns
import math

### Linear Algebra Recap

Linear algebra forms the backbone of most numerical simulations and machine learning algorithms. In this section, we briefly recall essential concepts such as vector spaces, matrix operations, inner products, and linear transformations — all of which serve as the mathematical foundation for the core functionalities of **tensorium**.

#### Vectors and Matrices

Vectors and matrices are the fundamental objects of linear algebra. They represent data structures for encoding points in space, transformations, and systems of equations. **tensorium** builds its core around efficient representations and operations on these objects, with a focus on performance and alignment.  

A matrix, in its most intrinsic mathematical sense, is a two-dimensional array of elements arranged in rows and columns. More formally, an $m \times n$ matrix is a function that assigns to each pair of indices $(i, j)$ a scalar entry $a_{ij}$, typically over a field such as $\mathbb{R}$ or $\mathbb{C}$. It can be seen as an element of the vector space $\mathbb{K}^{m \times n}$, where $\mathbb{K}$ denotes the base field.

In the special case where a matrix has only a single column (or row), it reduces to a vector, which we interpret as a point or direction in space. Vectors are thus a specific type of matrix, typically of size $n \times 1$ (column vector) or $1 \times n$ (row vector), and they inherit all structural and algebraic properties from the general matrix framework.  

Vectors can be represented either as column matrices in $\mathbb{K}^{n \times 1}$ or as row matrices in $\mathbb{K}^{1 \times n}$.


#### Vector Operations

The set $\mathbb{R}^n$ of all real $n$-dimensional vectors, equipped with the standard vector addition defined by  
$$(\mathbf{u} + \mathbf{v})_i := u_i + v_i \quad \text{for all } 1 \leq i \leq n,$$  
forms an abelian group $(\mathbb{R}^n, +)$, since addition is associative, commutative, has a neutral element (the zero vector), and each element has an additive inverse.

As an example, let  
$$
\mathbf{u} = \begin{pmatrix}
u_1 \\
u_2 \\
\vdots \\
u_n
\end{pmatrix}, \quad
\mathbf{v} = \begin{pmatrix}
v_1 \\
v_2 \\
\vdots \\
v_n
\end{pmatrix}
$$  
be two vectors in $\mathbb{R}^n$. Then their sum $\mathbf{w} = \mathbf{u} + \mathbf{v} \in \mathbb{R}^n$ is given by:
$$
\mathbf{w} = \begin{pmatrix}
u_1 + v_1 \\
u_2 + v_2 \\
\vdots \\
u_n + v_n
\end{pmatrix}.
$$

Just like vector addition, vector subtraction is defined componentwise. Given two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$, their difference $\mathbf{w} = \mathbf{u} - \mathbf{v} \in \mathbb{R}^n$ is defined by:  
$$(\mathbf{u} - \mathbf{v})_i := u_i - v_i \quad \text{for all } 1 \leq i \leq n.$$

This operation is well-defined and satisfies properties similar to addition, such as associativity with scalar multiplication and the existence of an additive inverse.  
However, **vector subtraction is not commutative in general**:
$$
\mathbf{u} - \mathbf{v} \neq \mathbf{v} - \mathbf{u} \quad \text{unless } \mathbf{u} = \mathbf{v}.
$$

For example, the difference of two vectors is:
$$
\mathbf{w} = \mathbf{u} - \mathbf{v} = \begin{pmatrix}
u_1 - v_1 \\
u_2 - v_2 \\
\vdots \\
u_n - v_n
\end{pmatrix}.
$$

The **dot product** (or inner product) of two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ is defined by:
$$
\langle \mathbf{u}, \mathbf{v} \rangle := \sum_{i=1}^n u_i v_i.
$$

This operation is bilinear, symmetric, and positive-definite. It induces the standard Euclidean norm:
$$
\| \mathbf{u} \| = \sqrt{ \langle \mathbf{u}, \mathbf{u} \rangle } = \sqrt{ \sum_{i=1}^n u_i^2 }.
$$


Scalar multiplication — or the **scaling of a vector by a scalar** — modifies both the magnitude and, possibly, the direction of the vector.

The operation **scl** denotes the scaling of a vector by a scalar, formally defined as the action of the field $\mathbb{K}$ on the vector space $\mathbb{K}^n$, mapping $(\lambda, \mathbf{v}) \mapsto \lambda \mathbf{v}$ such that each component of $\mathbf{v}$ is multiplied by $\lambda$.


In [36]:
from tensorium import Vector, Matrix, tns

# Vector add
u = Vector([2., 3.])
v = Vector([5., 7.])
u = tns.add_vec(u, v)
print("Vector add:\n", u)

# Vector sub
u = Vector([2., 3.])
v = Vector([5., 7.])
u = tns.sub_vec(u, v)
print("Vector sub:\n", u)

# Vector scl
u = Vector([2., 3.])
u = tns.scl_vec(u, 2.)
print("Vector scl:\n", u)


Vector add:
 [7, 10]
Vector sub:
 [-3, -4]
Vector scl:
 [4, 6]


#### Matrix Operations

The set $\mathbb{R}^{m \times n}$ of all real $m \times n$ matrices, equipped with the standard matrix addition defined by  
$$(A + B)_{ij} := A_{ij} + B_{ij} \quad \text{for all } 1 \leq i \leq m,\, 1 \leq j \leq n,$$  
forms an abelian group $(\mathbb{R}^{m \times n}, +)$, since addition is associative, commutative, has a neutral element (the zero matrix), and each element has an additive inverse.

As an example, we can therefore express the addition of matrices as:

Let  
$$A_{ij} = \begin{pmatrix}
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} & a_{m2} & \cdots & a_{mn}
\end{pmatrix}, \quad
B_{ij} = \begin{pmatrix}
b_{11} & b_{12} & \cdots & b_{1n} \\
b_{21} & b_{22} & \cdots & b_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
b_{m1} & b_{m2} & \cdots & b_{mn}
\end{pmatrix}$$  
be two matrices in $\mathbb{R}^{m \times n}$. Then their sum $C = A + B \in \mathbb{R}^{m \times n}$ is given by:  
$$
C_{ij} = \begin{pmatrix}
a_{11} + b_{11} & a_{12} + b_{12} & \cdots & a_{1n} + b_{1n} \\
a_{21} + b_{21} & a_{22} + b_{22} & \cdots & a_{2n} + b_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} + b_{m1} & a_{m2} + b_{m2} & \cdots & a_{mn} + b_{mn}
\end{pmatrix}.
$$

Just like matrix addition, matrix subtraction is defined componentwise. Given two matrices $A, B \in \mathbb{R}^{m \times n}$, their difference $C = A - B \in \mathbb{R}^{m \times n}$ is defined by:  
$$(A - B)_{ij} := A_{ij} - B_{ij} \quad \text{for all } 1 \leq i \leq m,\, 1 \leq j \leq n.$$

This operation is well-defined and satisfies properties similar to addition, such as associativity with respect to scalar multiplication and the existence of an additive inverse. However, **matrix subtraction is not commutative in general**:  
$$
A - B \neq B - A \quad \text{unless } A = B.
$$

For example, the difference of two matrices is given by:  
$$
C_{ij} = A_{ij} - B_{ij} = \begin{pmatrix}
a_{11} - b_{11} & a_{12} - b_{12} & \cdots & a_{1n} - b_{1n} \\
a_{21} - b_{21} & a_{22} - b_{22} & \cdots & a_{2n} - b_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} - b_{m1} & a_{m2} - b_{m2} & \cdots & a_{mn} - b_{mn}
\end{pmatrix}.
$$




In [37]:

# Matrix add
u = Matrix(2, 2)
u.fill([
    [1., 2.],
    [3., 4.]
])
v = Matrix(2, 2)
v.fill([
    [7., 4.],
    [-2., 2.]
])
u_add = tns.add_mat(u, v)
print("Matrix add:\n", u_add)

# Matrix sub
u = Matrix(2, 2)
u.fill([
    [1., 2.],
    [3., 4.]
])
v = Matrix(2, 2)
v.fill([
    [7., 4.],
    [-2., 2.]
])
u_sub = tns.sub_mat(u, v)
print("Matrix sub:\n", u_sub)

# Matrix scl
u = Matrix(2, 2)
u.fill([
    [1., 2.],
    [3., 4.]
])
u_scl = tns.scl_mat(u, 2.)
print("Matrix scl:\n", u_scl)


Matrix add:
 [
  [8, 6],
  [1, 6]
]
Matrix sub:
 [
  [-6, -2],
  [5, 2]
]
Matrix scl:
 [
  [2, 4],
  [6, 8]
]


#### Linear Combinations

Let $\mathbb{K}$ be a field (typically $\mathbb{R}$ or $\mathbb{C}$), and let $V$ be a vector space over $\mathbb{K}$.  
Given a finite set of vectors $\{\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_k\} \subset V$ and scalars $\lambda_1, \lambda_2, \dots, \lambda_k \in \mathbb{K}$, a **linear combination** of these vectors is an expression of the form:
$$
\mathbf{u} = \lambda_1 \mathbf{v}_1 + \lambda_2 \mathbf{v}_2 + \cdots + \lambda_k \mathbf{v}_k = \sum_{i=1}^{k} \lambda_i \mathbf{v}_i.
$$

The vector $\mathbf{u} \in V$ obtained this way lies in the **span** of $\{\mathbf{v}_1, \dots, \mathbf{v}_k\}$, denoted by:
$$
\text{Span}(\mathbf{v}_1, \dots, \mathbf{v}_k) := \left\{ \sum_{i=1}^k \lambda_i \mathbf{v}_i \,\middle|\, \lambda_i \in \mathbb{K} \right\}.
$$

Linear combinations are foundational in linear algebra, as they define:
- the **structure** of vector spaces,
- the concept of **linear dependence** (a set of vectors is linearly dependent if at least one is a linear combination of the others),
- and the construction of **bases**.

In computational contexts (e.g., numerical methods or machine learning), linear combinations frequently appear in the form of:
- updates to solution vectors,
- weighted sums,
- or transformations in lower-dimensional subspaces.

For example, if  
$$
\mathbf{v}_1 = \begin{pmatrix}1 \\ 2 \\ 3\end{pmatrix}, \quad \mathbf{v}_2 = \begin{pmatrix}-1 \\ 0 \\ 1\end{pmatrix}, \quad \lambda_1 = 2, \quad \lambda_2 = 3,
$$  
then the linear combination is:
$$
\mathbf{u} = 2 \mathbf{v}_1 + 3 \mathbf{v}_2 = \begin{pmatrix}2 \\ 4 \\ 6\end{pmatrix} + \begin{pmatrix}-3 \\ 0 \\ 3\end{pmatrix} = \begin{pmatrix}-1 \\ 4 \\ 9\end{pmatrix}.
$$


In [38]:
from tensorium import Vector, tns

# Base vectors
e1 = Vector([1.0, 0.0, 0.0])
e2 = Vector([0.0, 1.0, 0.0])
e3 = Vector([0.0, 0.0, 1.0])

# Arbitrary vectors
v1 = Vector([1.0, 2.0, 3.0])
v2 = Vector([0.0, 10.0, -100.0])

# Test 1
result1 = tns.linear_comb([e1, e2, e3], [10.0, -2.0, 0.5])
print("Test 1 Result (e1,e2,e3):", result1)

# Test 2
result2 = tns.linear_comb([v1, v2], [10.0, -2.0])
print("Test 2 Result (v1,v2):", result2)



Test 1 Result (e1,e2,e3): [10, -2, 0.5]
Test 2 Result (v1,v2): [10, 0, 230]


#### Linear Interpolation (lerp)

Linear interpolation, commonly abbreviated as **lerp**, is a fundamental operation that constructs a point along the line segment between two vectors. Given two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{K}^n$ and a scalar parameter $t \in [0, 1]$, the linear interpolation between $\mathbf{u}$ and $\mathbf{v}$ is defined as:
$$
\text{lerp}(\mathbf{u}, \mathbf{v}; t) := (1 - t)\mathbf{u} + t\mathbf{v}.
$$

This operation yields a new vector that lies on the segment joining $\mathbf{u}$ and $\mathbf{v}$:
- If $t = 0$, then $\text{lerp}(\mathbf{u}, \mathbf{v}; 0) = \mathbf{u}$.
- If $t = 1$, then $\text{lerp}(\mathbf{u}, \mathbf{v}; 1) = \mathbf{v}$.
- If $0 < t < 1$, the result is a **convex combination** of $\mathbf{u}$ and $\mathbf{v}$.

Lerp is a special case of a **linear combination** where the weights are $(1 - t)$ and $t$, and they always sum to 1. It is widely used in:
- computer graphics and animation (for motion interpolation),
- numerical simulations (for blending values),
- and optimization algorithms (for parameter continuation).

Geometrically, the result $\mathbf{w} = \text{lerp}(\mathbf{u}, \mathbf{v}; t)$ is the unique point on the straight line segment between $\mathbf{u}$ and $\mathbf{v}$, at a relative position $t$ from $\mathbf{u}$ toward $\mathbf{v}$.


In [39]:
from tensorium import Vector, Matrix, tns

print(tns.lerp(Vector([0.0]), Vector([1.0]), 0.0))   # [0.0]
print(tns.lerp(Vector([0.0]), Vector([1.0]), 1.0))   # [1.0]
print(tns.lerp(Vector([0.0]), Vector([1.0]), 0.5))   # [0.5]
print(tns.lerp(Vector([21.0]), Vector([42.0]), 0.3)) # [27.3]

v1 = Vector([2.0, 1.0])
v2 = Vector([4.0, 2.0])
print(tns.lerp(v1, v2, 0.3))  # [2.6, 1.3]

m1 = Matrix(2, 2)
m1.fill([[2.0, 1.0], [3.0, 4.0]])

m2 = Matrix(2, 2)
m2.fill([[20.0, 10.0], [30.0, 40.0]])

print(tns.lerp_mat(m1, m2, 0.5))


[0]
[1]
[0.5]
[27.3]
[2.6, 1.3]
[
  [11, 5.5],
  [16.5, 22]
]


#### Dot Product

The **dot product** (also known as the inner product in $\mathbb{R}^n$) is a bilinear, symmetric operation that maps two vectors to a scalar. Given two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$, their dot product is defined as:
$$
\mathbf{u} \cdot \mathbf{v} := \sum_{i=1}^{n} u_i v_i.
$$

This operation satisfies the following key properties:
- **Commutativity**: $\mathbf{u} \cdot \mathbf{v} = \mathbf{v} \cdot \mathbf{u}$
- **Bilinearity**:
  $$
  (\lambda \mathbf{u} + \mu \mathbf{v}) \cdot \mathbf{w} = \lambda (\mathbf{u} \cdot \mathbf{w}) + \mu (\mathbf{v} \cdot \mathbf{w}), \quad \forall \lambda, \mu \in \mathbb{R}
  $$
- **Positive-definiteness**: $\mathbf{u} \cdot \mathbf{u} \geq 0$, and $\mathbf{u} \cdot \mathbf{u} = 0$ if and only if $\mathbf{u} = \mathbf{0}$

The dot product induces the **Euclidean norm**:
$$
\| \mathbf{u} \| := \sqrt{\mathbf{u} \cdot \mathbf{u}} = \left( \sum_{i=1}^{n} u_i^2 \right)^{1/2}
$$

It also encodes geometric information, such as the **angle** $\theta$ between two vectors:
$$
\mathbf{u} \cdot \mathbf{v} = \| \mathbf{u} \| \, \| \mathbf{v} \| \cos \theta
$$
Thus:
- If $\mathbf{u} \cdot \mathbf{v} > 0$, the angle between them is acute.
- If $\mathbf{u} \cdot \mathbf{v} = 0$, the vectors are orthogonal.
- If $\mathbf{u} \cdot \mathbf{v} < 0$, the angle is obtuse.

The dot product is fundamental in linear algebra, geometry, and physics, and is central to algorithms in projection, optimization, and similarity computations.


In [40]:
from tensorium import Vector, tns

print("\nDot product = \n")

u1 = Vector([0.0, 0.0])
v1 = Vector([1.0, 1.0])
print("u1 . v1 =", tns.dot_vec(u1, v1))  # 0.0

u2 = Vector([1.0, 1.0])
v2 = Vector([1.0, 1.0])
print("u2 . v2 =", tns.dot_vec(u2, v2))  # 2.0

u3 = Vector([-1.0, 6.0])
v3 = Vector([3.0, 2.0])
print("u3 . v3 =", tns.dot_vec(u3, v3))  # 9.0



Dot product = 

u1 . v1 = 0.0
u2 . v2 = 2.0
u3 . v3 = 9.0


#### Vector Norms

Given a vector $\mathbf{v} = (v_1, v_2, \dots, v_n) \in \mathbb{K}^n$ (with $\mathbb{K} = \mathbb{R}$ or $\mathbb{C}$), a **norm** is a function $\|\cdot\| : \mathbb{K}^n \rightarrow \mathbb{R}$ that satisfies:

- **Non-negativity**: $\|\mathbf{v}\| \geq 0$
- **Definiteness**: $\|\mathbf{v}\| = 0 \iff \mathbf{v} = \mathbf{0}$
- **Homogeneity**: $\|\lambda \mathbf{v}\| = |\lambda| \cdot \|\mathbf{v}\|$
- **Triangle inequality**: $\|\mathbf{u} + \mathbf{v}\| \leq \|\mathbf{u}\| + \|\mathbf{v}\|$

Below are the most commonly used norms:

---

**1. L¹ norm (Manhattan norm)**

Also called the **taxicab norm**, it is defined as the sum of the absolute values of the components:
$$
\|\mathbf{v}\|_1 := \sum_{i=1}^{n} |v_i|.
$$

Example:
$$
\mathbf{v} = \begin{pmatrix} 3 \\ -7 \\ 2 \end{pmatrix}, \quad \|\mathbf{v}\|_1 = |3| + |{-7}| + |2| = 12.
$$

---

**2. L² norm (Euclidean norm)**

This is the standard notion of length in Euclidean space, induced by the dot product:
$$
\|\mathbf{v}\|_2 := \sqrt{\sum_{i=1}^{n} |v_i|^2} = \sqrt{\mathbf{v} \cdot \mathbf{v}}.
$$

Example:
$$
\mathbf{v} = \begin{pmatrix} 3 \\ -7 \\ 2 \end{pmatrix}, \quad \|\mathbf{v}\|_2 = \sqrt{3^2 + (-7)^2 + 2^2} = \sqrt{62}.
$$

---

**3. L∞ norm (Infinity norm)**

Also called the **maximum norm**, it is the largest absolute component of the vector:
$$
\|\mathbf{v}\|_{\infty} := \max_{1 \leq i \leq n} |v_i|.
$$

Example:
$$
\mathbf{v} = \begin{pmatrix} 3 \\ -7 \\ 2 \end{pmatrix}, \quad \|\mathbf{v}\|_{\infty} = \max\{3, 7, 2\} = 7.
$$

---

These norms are all special cases of the **p-norm**:
$$
\|\mathbf{v}\|_p := \left( \sum_{i=1}^{n} |v_i|^p \right)^{1/p}, \quad p \geq 1.
$$
With:
- $p = 1 \Rightarrow$ L¹ norm
- $p = 2 \Rightarrow$ L² norm
- $p \to \infty \Rightarrow$ L∞ norm


In [41]:
from tensorium import Vector, tns

print("\nNorms tests:\n")

u0 = Vector([0.0, 0.0, 0.0])
print("u0 =", u0)
print("norm_1 =", tns.norm_1(u0), ", norm_2 =", tns.norm_2(u0), ", norm_inf =", tns.norm_inf(u0))

u1 = Vector([1.0, 2.0, 3.0])
print("u1 =", u1)
print("norm_1 =", tns.norm_1(u1), ", norm_2 =", tns.norm_2(u1), ", norm_inf =", tns.norm_inf(u1))

u2 = Vector([-1.0, -2.0])
print("u2 =", u2)
print("norm_1 =", tns.norm_1(u2), ", norm_2 =", tns.norm_2(u2), ", norm_inf =", tns.norm_inf(u2))




Norms tests:

u0 = [0, 0, 0]
norm_1 = 0.0 , norm_2 = 0.0 , norm_inf = 0.0
u1 = [1, 2, 3]
norm_1 = 6.0 , norm_2 = 3.7416574954986572 , norm_inf = 3.0
u2 = [-1, -2]
norm_1 = 3.0 , norm_2 = 2.2360680103302 , norm_inf = 2.0


#### Cosine Similarity

The **cosine similarity** between two non-zero vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ is defined as the cosine of the angle $\theta$ between them:
$$
\cos(\theta) := \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \, \|\mathbf{v}\|_2}.
$$

This metric measures the **directional alignment** between $\mathbf{u}$ and $\mathbf{v}$, independently of their magnitudes.

**Properties:**
- $\cos(\theta) \in [-1, 1]$
- $\cos(\theta) = 1$ if $\mathbf{u}$ and $\mathbf{v}$ point in the **same direction**
- $\cos(\theta) = 0$ if $\mathbf{u}$ and $\mathbf{v}$ are **orthogonal**
- $\cos(\theta) = -1$ if they point in **opposite directions**

**Applications**:
- Frequently used in **machine learning**, **information retrieval**, and **natural language processing** to quantify similarity between high-dimensional vectors (e.g., document embeddings).
- In contrast to the Euclidean distance, cosine similarity is **scale-invariant**: multiplying either vector by a positive scalar does not affect the result.

**Example**:  
Let  
$$
\mathbf{u} = \begin{pmatrix}1 \\ 0\end{pmatrix}, \quad \mathbf{v} = \begin{pmatrix}1 \\ 1\end{pmatrix},
$$  
then
$$
\cos(\theta) = \frac{1 \cdot 1 + 0 \cdot 1}{\sqrt{1^2 + 0^2} \cdot \sqrt{1^2 + 1^2}} = \frac{1}{\sqrt{2}} \approx 0.707.
$$

This corresponds to an angle of $\theta = \frac{\pi}{4} = 45^\circ$.


In [42]:
print("\nCosine tests:\n")

u = Vector([1.0, 0.0])
v = Vector([1.0, 0.0])
print("cosine([1, 0], [1, 0]) =", tns.cosine(u, v))  # 1.0

u = Vector([1.0, 0.0])
v = Vector([0.0, 1.0])
print("cosine([1, 0], [0, 1]) =", tns.cosine(u, v))  # 0.0

u = Vector([-1.0, 1.0])
v = Vector([1.0, -1.0])
print("cosine([-1, 1], [1, -1]) =", tns.cosine(u, v))  # -1.0

u = Vector([2.0, 1.0])
v = Vector([4.0, 2.0])
print("cosine([2, 1], [4, 2]) =", tns.cosine(u, v))  # 1.0

u = Vector([1.0, 2.0, 3.0])
v = Vector([4.0, 5.0, 6.0])
print("cosine([1,2,3], [4,5,6]) =", tns.cosine(u, v))




Cosine tests:

cosine([1, 0], [1, 0]) = 1.0
cosine([1, 0], [0, 1]) = 0.0
cosine([-1, 1], [1, -1]) = -1.0000001192092896
cosine([2, 1], [4, 2]) = 1.0
cosine([1,2,3], [4,5,6]) = 0.9746317863464355


#### Cross Product (3D Only)

The **cross product** is a binary operation defined only in $\mathbb{R}^3$, which takes two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^3$ and returns a third vector $\mathbf{w} = \mathbf{u} \times \mathbf{v} \in \mathbb{R}^3$ that is orthogonal to both $\mathbf{u}$ and $\mathbf{v}$.

**Definition**:  
Let  
$$
\mathbf{u} = \begin{pmatrix} u_1 \\ u_2 \\ u_3 \end{pmatrix}, \quad
\mathbf{v} = \begin{pmatrix} v_1 \\ v_2 \\ v_3 \end{pmatrix},
$$  
then the cross product is defined as:
$$
\mathbf{u} \times \mathbf{v} = \begin{pmatrix}
u_2 v_3 - u_3 v_2 \\
u_3 v_1 - u_1 v_3 \\
u_1 v_2 - u_2 v_1
\end{pmatrix}.
$$

**Properties**:
- $\mathbf{u} \times \mathbf{v}$ is orthogonal to both $\mathbf{u}$ and $\mathbf{v}$
- $\|\mathbf{u} \times \mathbf{v}\| = \|\mathbf{u}\| \cdot \|\mathbf{v}\| \cdot \sin(\theta)$, where $\theta$ is the angle between $\mathbf{u}$ and $\mathbf{v}$
- **Anti-symmetric**: $\mathbf{u} \times \mathbf{v} = -(\mathbf{v} \times \mathbf{u})$
- $\mathbf{u} \times \mathbf{u} = \mathbf{0}$

Geometrically, the cross product gives a vector perpendicular to the plane defined by $\mathbf{u}$ and $\mathbf{v}$, with orientation determined by the **right-hand rule**.

**Example**:  
Let  
$$
\mathbf{u} = \begin{pmatrix}1 \\ 0 \\ 0\end{pmatrix}, \quad
\mathbf{v} = \begin{pmatrix}0 \\ 1 \\ 0\end{pmatrix},
$$  
then  
$$
\mathbf{u} \times \mathbf{v} = \begin{pmatrix}0 \\ 0 \\ 1\end{pmatrix}.
$$


In [43]:
from tensorium import Vector, tns

print("\nCross product tests:\n")

u = Vector([0.0, 0.0, 1.0])
v = Vector([1.0, 0.0, 0.0])
print("cross([0,0,1], [1,0,0]) =", tns.cross(u, v))

u = Vector([1.0, 2.0, 3.0])
v = Vector([4.0, 5.0, 6.0])
print("cross([1,2,3], [4,5,6]) =", tns.cross(u, v))

u = Vector([4.0, 2.0, -3.0])
v = Vector([-2.0, -5.0, 16.0])
print("cross([4,2,-3], [-2,-5,16]) =", tns.cross(u, v))



Cross product tests:

cross([0,0,1], [1,0,0]) = [0, 1, 0]
cross([1,2,3], [4,5,6]) = [-3, 6, -3]
cross([4,2,-3], [-2,-5,16]) = [17, -58, -16]


### Matrix product

The product of two matrices is, in general, not commutative; that is, $AB \neq BA$ in general.  
However, matrix multiplication is associative and distributive over matrix addition, by virtue of the algebraic rules governing this operation.

For a matrix product, let $A \in \mathbb{R}^{m \times p}$ and $B \in \mathbb{R}^{p \times n}$.  
The product $C = AB \in \mathbb{R}^{m \times n}$ is defined by:  
$$
C_{ij} := \sum_{k=1}^{p} A_{ik} B_{kj}, \quad \text{for all } 1 \leq i \leq m,\, 1 \leq j \leq n.
$$

This definition ensures that the matrix product is well-defined only when the number of columns of $A$ equals the number of rows of $B$. The resulting matrix $C$ has the same number of rows as $A$ and the same number of columns as $B$.

Let $A \in \mathbb{R}^{m \times p}$ and $B \in \mathbb{R}^{p \times n}$ be two matrices, defined as:

$$
A = \begin{pmatrix}
a_{11} & a_{12} & \cdots & a_{1p} \\
a_{21} & a_{22} & \cdots & a_{2p} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} & a_{m2} & \cdots & a_{mp}
\end{pmatrix}, \quad
B = \begin{pmatrix}
b_{11} & b_{12} & \cdots & b_{1n} \\
b_{21} & b_{22} & \cdots & b_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
b_{p1} & b_{p2} & \cdots & b_{pn}
\end{pmatrix}.
$$

Then their product $C = AB \in \mathbb{R}^{m \times n}$ is the matrix:

$$
C = \begin{pmatrix}
\sum_{k=1}^{p} a_{1k} b_{k1} & \sum_{k=1}^{p} a_{1k} b_{k2} & \cdots & \sum_{k=1}^{p} a_{1k} b_{kn} \\
\sum_{k=1}^{p} a_{2k} b_{k1} & \sum_{k=1}^{p} a_{2k} b_{k2} & \cdots & \sum_{k=1}^{p} a_{2k} b_{kdn} \\
\vdots & \vdots & \ddots & \vdots \\
\sum_{k=1}^{p} a_{mk} b_{k1} & \sum_{k=1}^{p} a_{mk} b_{k2} & \cdots & \sum_{k=1}^{p} a_{mk} b_{kn}
\end{pmatrix}.
$$

Matrix multiplication is **associative** and **distributive** over addition:  
$$
A(BC) = (AB)C, \quad A(B + C) = AB + AC, \quad (A + B)C = AC + BC.
$$

**Linearity with respect to rows and columns.**  
Let $A = (a_{ik})$, $B = (b_{kj})$, and $C = AB$. Then each entry $C_{ij}$ is obtained as the dot product of the $i$-th row of $A$ and the $j$-th column of $B$:  
$$
C_{ij} = \text{row}_i(A) \cdot \text{col}_j(B).
$$

In [44]:
from tensorium import Matrix, Vector, tns

print("\nMultiplication tests:\n")

u = Matrix(2, 2)
u.fill([[1.0, 0.0],
        [0.0, 1.0]])
v = Vector([4.0, 2.0])
print("u.mul_vec(v) =", tns.mul_vec(u, v))  # [4.0, 2.0]

u = Matrix(2, 2)
u.fill([[2.0, 0.0],
        [0.0, 2.0]])
v = Vector([4.0, 2.0])
print("u.mul_vec(v) =", tns.mul_vec(u, v))  # [8.0, 4.0]

u = Matrix(2, 2)
u.fill([[2.0, -2.0],
        [-2.0, 2.0]])
v = Vector([4.0, 2.0])
print("u.mul_vec(v) =", tns.mul_vec(u, v))  # [4.0, -4.0]

u = Matrix(2, 2)
v = Matrix(2, 2)
u.fill([[1.0, 0.0],
        [0.0, 1.0]])
v.fill([[1.0, 0.0],
        [0.0, 1.0]])
print("u.mul(v) =", tns.mul(u, v))  # [[1.0, 0.0], [0.0, 1.0]]

u = Matrix(2, 2)
v = Matrix(2, 2)
u.fill([[1.0, 0.0],
        [0.0, 1.0]])
v.fill([[2.0, 1.0],
        [4.0, 2.0]])
print("u.mul(v) =", tns.mul(u, v))  # [[2.0, 1.0], [4.0, 2.0]]

u = Matrix(2, 2)
v = Matrix(2, 2)
u.fill([[3.0, -5.0],
        [6.0, 8.0]])
v.fill([[2.0, 1.0],
        [4.0, 2.0]])
print("u.mul(v) =", tns.mul(u, v))  # [[-14.0, -7.0], [44.0, 22.0]]



Multiplication tests:

u.mul_vec(v) = [4, 2]
u.mul_vec(v) = [8, 4]
u.mul_vec(v) = [4, -4]
u.mul(v) = [
  [1, 0],
  [0, 1]
]
u.mul(v) = [
  [2, 1],
  [4, 2]
]
u.mul(v) = [
  [-14, -7],
  [44, 22]
]


#### Trace of a Matrix

The **trace** of a square matrix is defined as the sum of its diagonal elements.  
Let $A \in \mathbb{K}^{n \times n}$ be a square matrix over a field $\mathbb{K}$ (typically $\mathbb{R}$ or $\mathbb{C}$), then the **trace** of $A$ is given by:
$$
\mathrm{Tr}(A) := \sum_{i=1}^{n} A_{ii}.
$$

That is, the trace extracts the sum of the entries along the main diagonal of the matrix.

**Properties**:
- $\mathrm{Tr}(A + B) = \mathrm{Tr}(A) + \mathrm{Tr}(B)$
- $\mathrm{Tr}(\lambda A) = \lambda \cdot \mathrm{Tr}(A)$ for any scalar $\lambda$
- $\mathrm{Tr}(A^T) = \mathrm{Tr}(A)$
- $\mathrm{Tr}(AB) = \mathrm{Tr}(BA)$ (cyclic property, valid when dimensions match)
- The trace is **linear** and **basis-independent**

**Example**:  
If  
$$
A = \begin{pmatrix}
1 & 2 & 3 \\
0 & 4 & 5 \\
0 & 0 & 6
\end{pmatrix}, \quad \mathrm{Tr}(A) = 1 + 4 + 6 = 11.
$$


In [45]:
from tensorium import Matrix, tns

print("\nTrace tests:\n")

u = Matrix(2, 2)
u.fill([
    [1.0, 0.0],
    [0.0, 1.0],
])
print("Trace of identity matrix =", tns.trace_mat(u))  # 2.0

u = Matrix(3, 3)
u.fill([
    [2.0, -5.0, 0.0],
    [4.0, 3.0, 7.0],
    [-2.0, 3.0, 4.0],
])
print("Trace of matrix =", tns.trace_mat(u))  # 2 + 3 + 4 = 9.0

u = Matrix(3, 3)
u.fill([
    [-2.0, -8.0, 4.0],
    [1.0, -23.0, 4.0],
    [0.0, 6.0, 4.0],
])
print("Trace of matrix =", tns.trace_mat(u))  # -2 + (-23) + 4 = -21.0



Trace tests:

Trace of identity matrix = [
  [2]
]
Trace of matrix = [
  [9]
]
Trace of matrix = [
  [-21]
]


#### Transpose of a Matrix

The **transpose** of a matrix is an operation that flips a matrix over its diagonal, exchanging its rows with its columns.  
Let $A \in \mathbb{K}^{m \times n}$ be a matrix over a field $\mathbb{K}$, then its **transpose**, denoted $A^T$, is defined by:
$$
(A^T)_{ij} := A_{ji}, \quad \text{for all } 1 \leq i \leq n,\ 1 \leq j \leq m.
$$

That is, if
$$
A = \begin{pmatrix}
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} & a_{m2} & \cdots & a_{mn}
\end{pmatrix},
$$  
then
$$
A^T = \begin{pmatrix}
a_{11} & a_{21} & \cdots & a_{m1} \\
a_{12} & a_{22} & \cdots & a_{m2} \\
\vdots & \vdots & \ddots & \vdots \\
a_{1n} & a_{2n} & \cdots & a_{mn}
\end{pmatrix}.
$$

---

**Properties**:
- $(A^T)^T = A$ (involution)
- $(A + B)^T = A^T + B^T$
- $(\lambda A)^T = \lambda A^T$ for any scalar $\lambda$
- $(AB)^T = B^T A^T$ (reverses the product order)
- $\mathrm{Tr}(A^T) = \mathrm{Tr}(A)$

The transpose plays a fundamental role in linear algebra, particularly in defining symmetric matrices, adjoints, and orthogonality.

---

**Example**:  
Let  
$$
A = \begin{pmatrix}
1 & 2 & 3 \\
4 & 5 & 6
\end{pmatrix}, \quad
A^T = \begin{pmatrix}
1 & 4 \\
2 & 5 \\
3 & 6
\end{pmatrix}.
$$


In [46]:
A = Matrix(2, 3)
A.fill([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

AT = tns.transpose_mat(A)

print("A =\n", A)
print("A^T =\n", AT)


A =
 [
  [1, 2, 3],
  [4, 5, 6]
]
A^T =
 [
  [1, 4],
  [2, 5],
  [3, 6]
]


#### (interlude) Solving Linear Systems: Jacobi Method

The **Jacobi method** is an iterative algorithm used to solve a system of linear equations of the form:
$$
A \mathbf{x} = \mathbf{b},
$$
where $A \in \mathbb{R}^{n \times n}$ is a square matrix, $\mathbf{x}, \mathbf{b} \in \mathbb{R}^n$ are vectors.

---

**Idea**:  
Decompose the matrix $A$ into:
- $D$ = diagonal part of $A$
- $R$ = remainder (strictly off-diagonal part): $R = A - D$

Then rewrite the system as:
$$
\mathbf{x} = D^{-1}(\mathbf{b} - R \mathbf{x}).
$$

This yields the **Jacobi iteration**:
$$
\mathbf{x}^{(k+1)} = D^{-1}(\mathbf{b} - R \mathbf{x}^{(k)}),
$$
or component-wise:
$$
x_i^{(k+1)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j \ne i} a_{ij} x_j^{(k)} \right), \quad \text{for } i = 1, \dots, n.
$$

---

**Convergence Criteria**:
- The method converges if $A$ is **strictly diagonally dominant**, i.e.,
$$
|a_{ii}| > \sum_{j \ne i} |a_{ij}| \quad \text{for all } i.
$$
- Alternatively, convergence is guaranteed if $A$ is **symmetric positive definite**.

---

**Example**:  
Let
$$
A = \begin{pmatrix}
4 & 1 & 2 \\
3 & 5 & 1 \\
1 & 1 & 3
\end{pmatrix}, \quad
\mathbf{b} = \begin{pmatrix} 4 \\ 7 \\ 3 \end{pmatrix}
$$  
Starting with $\mathbf{x}^{(0)} = \mathbf{0}$, we compute iteratively using:
$$
x_1^{(k+1)} = \frac{1}{4} \left( 4 - x_2^{(k)} - 2x_3^{(k)} \right) \\
x_2^{(k+1)} = \frac{1}{5} \left( 7 - 3x_1^{(k)} - x_3^{(k)} \right) \\
x_3^{(k+1)} = \frac{1}{3} \left( 3 - x_1^{(k)} - x_2^{(k)} \right)
$$

---

**Stopping Criterion**:  
The iteration stops when:
$$
\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| < \varepsilon,
$$  
for a given tolerance $\varepsilon > 0$.



In [47]:
A = Matrix(3, 3)
A.fill([
    [4.0, 1.0, 2.0],
    [3.0, 5.0, 1.0],
    [1.0, 1.0, 3.0]
])
b = Vector([4.0, 7.0, 3.0])

x = tns.jacobi_solve(A, b, tol=1e-6, max_iter=100)

print("Solution x =", x)


Solution x = [0.5, 1, 0.5]


#### Solving Linear Systems: Gaussian Elimination

**Gaussian elimination** is a direct method to solve a linear system of the form:
$$
A \mathbf{x} = \mathbf{b}, \quad \text{with } A \in \mathbb{R}^{n \times n},\ \mathbf{b} \in \mathbb{R}^n.
$$

It consists of two main steps:
1. **Forward elimination**: Transform the augmented matrix $[A \mid \mathbf{b}]$ into an upper triangular form $[U \mid \mathbf{c}]$.
2. **Back substitution**: Solve the triangular system $U \mathbf{x} = \mathbf{c}$ starting from the last row up to the first.

---

**Step 1 – Forward Elimination**  
Eliminate the coefficients below the diagonal by replacing row $i$ with:
$$
R_i \leftarrow R_i - \frac{a_{ij}}{a_{jj}} R_j, \quad \text{for } i > j.
$$

This operation zeroes out the lower-triangular entries to produce an upper-triangular matrix $U$.

---

**Step 2 – Back Substitution**  
Once the matrix is in the form:
$$
U = \begin{pmatrix}
u_{11} & u_{12} & \cdots & u_{1n} \\
0 & u_{22} & \cdots & u_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & u_{nn}
\end{pmatrix}, \quad
\mathbf{c} = \begin{pmatrix} c_1 \\ c_2 \\ \vdots \\ c_n \end{pmatrix},
$$  
solve the system starting from:
$$
x_n = \frac{c_n}{u_{nn}}, \quad x_{n-1} = \frac{c_{n-1} - \sum_{j=n-1+1}^{n} u_{n-1,j} x_j}{u_{n-1,n-1}}, \quad \dots
$$

---

**Requirements**:
- Pivot elements $a_{ii} \ne 0$ (unless partial pivoting is used)
- Works for all non-singular square systems

**Example**:  
Solve:
$$
A = \begin{pmatrix}
2 & 1 & -1 \\
-3 & -1 & 2 \\
-2 & 1 & 2
\end{pmatrix}, \quad
\mathbf{b} = \begin{pmatrix}
8 \\
-11 \\
-3
\end{pmatrix}
$$

After forward elimination and back substitution, the solution is:
$$
\mathbf{x} = \begin{pmatrix} 2 \\ 3 \\ -1 \end{pmatrix}
$$


In [48]:
from tensorium import Matrix, Vector, tns

A = Matrix(3, 3)
A.fill([
    [2.0, 1.0, -1.0],
    [-3.0, -1.0, 2.0],
    [-2.0, 1.0, 2.0]
])
b = Vector([8.0, -11.0, -3.0])

x = tns.gauss_solve(A, b)

print("Solution x =", x)


Solution x = [2, 3, -1]


#### Row Echelon Form of a Matrix

The **row echelon form (REF)** of a matrix is a canonical form obtained by a finite sequence of **elementary row operations** applied to a matrix over a field $\mathbb{K}$. It is a foundational concept in linear algebra, particularly in the resolution of linear systems and rank computation.

---

**Definition**:

Let $A \in \mathbb{K}^{m \times n}$ be a matrix. We say that $A$ is in **row echelon form** if it satisfies the following three conditions:

1. **All zero rows (if any) appear at the bottom** of the matrix.  
2. In each nonzero row, the first nonzero entry (called a **pivot**) is strictly to the right of the pivot in the row above.  
3. **All entries below each pivot are zero.**

That is, there exists a strictly increasing sequence of pivot column indices $j_1 < j_2 < \dots < j_r$ such that:
- Row $i$ has pivot at column $j_i$,
- For all $k > i$, $A_{k j_i} = 0$.

---


**Example**:

Let the system $A x = b$ be given by:

$$
A = \begin{pmatrix}
1 & 2 & -1 \\
2 & 4 & -2 \\
-1 & -2 & 1
\end{pmatrix}, \quad
b = \begin{pmatrix}
3 \\
6 \\
-3
\end{pmatrix}
$$

This augmented system can be written as:
$$
[A \, | \, b] =
\begin{pmatrix}
1 & 2 & -1 & \big| & 3 \\
2 & 4 & -2 & \big| & 6 \\
-1 & -2 & 1 & \big| & -3
\end{pmatrix}
$$

We apply Gaussian elimination:

- $R_2 \leftarrow R_2 - 2 R_1$  
- $R_3 \leftarrow R_3 + R_1$

This yields:

$$
\begin{pmatrix}
1 & 2 & -1 & \big| & 3 \\
0 & 0 & 0 & \big| & 0 \\
0 & 0 & 0 & \big| & 0
\end{pmatrix}
$$


**Properties**:

- The row echelon form is **not unique**; it depends on the sequence of elementary row operations.
- Every matrix $A \in \mathbb{K}^{m \times n}$ is **row equivalent** to a matrix in row echelon form.
- The number of pivots equals the **rank** of the matrix.
- REF is stable under left multiplication by an invertible matrix: if $P A = U$ where $P$ is invertible and $U$ is in REF, then $A$ and $U$ are row equivalent.

---

**Notation**:

If $A \leadsto U$ denotes the transformation of $A$ into its row echelon form $U$ by a sequence of row operations, then:
$$
\exists P \in \mathrm{GL}_m(\mathbb{K}) \text{, } PA = U
$$
where $U$ is in row echelon form and $P$ is invertible.

---

**Remark**:

The row echelon form is a prerequisite to defining the **reduced row echelon form (RREF)**, in which:
- All pivots are equal to 1,
- Each pivot is the only nonzero entry in its column.

This further simplification requires back-substitution steps.


In [49]:
from tensorium import Matrix, Vector, tns

u1 = Matrix(3, 3)
u1.fill([
    [1., 0., 0.],
    [0., 1., 0.],
    [0., 0., 1.]
])
print("Test 1: Identity Matrix row echelon")
tns.row_echelon(u1)
print(u1)

u2 = Matrix(2, 2)
u2.fill([
    [1., 2.],
    [3., 4.]
])
print("Test 2: Full rank 2x2 Matrix row echelon")
tns.row_echelon(u2)
print(u2)

u3 = Matrix(2, 2)
u3.fill([
    [1., 2.],
    [2., 4.]
])
print("Test 3: Rank-deficient 2x2 Matrix row echelon")
tns.row_echelon(u3)
print(u3)

u4 = Matrix(3, 5)
u4.fill([
    [8., 5., -2., 4., 28.],
    [4., 2.5, 20., 4., -4.],
    [8., 5., 1., 4., 17.],
])
print("Test 4: 3x5 Matrix row echelon")
tns.row_echelon(u4)
print(u4)


Test 1: Identity Matrix row echelon
[
  [1, 0, 0],
  [0, 1, 0],
  [0, 0, 1]
]
Test 2: Full rank 2x2 Matrix row echelon
[
  [1, 0],
  [-0, 1]
]
Test 3: Rank-deficient 2x2 Matrix row echelon
[
  [1, 2],
  [0, 0]
]
Test 4: 3x5 Matrix row echelon
[
  [1, 0.625, 0, 0, -12.1667],
  [0, 0, 1, 0, -3.66667],
  [-0, -0, -0, 1, 29.5]
]


#### Determinant of a Matrix

The **determinant** is a scalar associated with a square matrix that encodes crucial properties such as invertibility, volume distortion under linear transformation, and orientation preservation. Let $A \in \mathbb{K}^{n \times n}$ be a square matrix over a field. The determinant of $A$, denoted $\det(A)$ or $|A|$, is defined recursively.

If $n = 1$, the matrix has a single entry and we define $\det(A) = A_{11}$.  
If $n \geq 2$, the determinant is computed by cofactor expansion along the first row:
$$
\det(A) = \sum_{j=1}^{n} (-1)^{1+j} A_{1j} \cdot \det(M_{1j}),
$$
where $M_{1j}$ is the minor matrix obtained by deleting row 1 and column $j$ of $A$.

---

**Example (3×3)**:  
Let
$$
A = \begin{pmatrix}
a & b & c \\
d & e & f \\
g & h & i
\end{pmatrix}.
$$
Then the determinant is given by the expression
$$
\det(A) = aei + bfg + cdh - afh - bdi - ceg.
$$

---

The determinant satisfies several important algebraic properties. For instance, we have $\det(I_n) = 1$, where $I_n$ is the identity matrix, and $\det(A^T) = \det(A)$, meaning it is invariant under transposition. It is multiplicative: $\det(AB) = \det(A) \cdot \det(B)$ for all square matrices of compatible size. If we scale the matrix by a scalar $\lambda$, then $\det(\lambda A) = \lambda^n \cdot \det(A)$. Finally, a matrix is invertible if and only if $\det(A) \ne 0$.

---

The determinant also responds in predictable ways to elementary row operations. Swapping two rows changes its sign. Multiplying a row by a scalar $\lambda$ multiplies the determinant by $\lambda$. However, adding a multiple of one row to another does not affect its value.

---

From a geometric perspective, the absolute value $|\det(A)|$ represents the **volume scaling factor** of the linear transformation defined by $A$, in $\mathbb{R}^2$, $\mathbb{R}^3$, or higher dimensions. A negative determinant indicates a **reversal of orientation**.

---

**Simple example**:  
Let
$$
A = \begin{pmatrix}
1 & 2 \\
3 & 4
\end{pmatrix}.
$$
Then we compute the determinant as $\det(A) = 1 \cdot 4 - 2 \cdot 3 = -2$.


In [50]:
from tensorium import Matrix, tns

Asin = Matrix(2, 2)
Asin.fill([
    [1, -1],
    [-1, 1]
])

Adiag = Matrix(3, 3)
Adiag.fill([
    [2., 0., 0.],
    [0., 2., 0.],
    [0., 0., 2.],
])
print("Determinant of Adiag =", tns.det_mat(Adiag))
Arand = Matrix(3, 3)
Arand.fill([
    [8., 5., -2.],
    [4., 7., 20.],
    [7., 6., 1.],
])
print("Determinant of Arand =", tns.det_mat(Arand))


print("Determinant of Asin =", tns.det_mat(Asin))
A = Matrix(2, 2)
A.fill([
    [1.0, 2.0],
    [3.0, 4.0]
])

print("Determinant of A =", tns.det_mat(A))


Abig = Matrix(4, 4)
Abig.fill([
    [ 8., 5., -2., 4.],
    [ 4., 2.5, 20., 4.],
    [ 8., 5., 1., 4.],
    [28., -4., 17., 1.],
])

print("Determinant of Abig =", tns.det_mat(Abig))


Determinant of Adiag = 8.0
Determinant of Arand = -173.99998474121094
Determinant of Asin = 0.0
Determinant of A = -1.9999998807907104
Determinant of Abig = 1031.9998779296875


#### Inverse of a Matrix

Given a square matrix $A \in \mathbb{K}^{n \times n}$ over a field $\mathbb{K}$, the **inverse** of $A$, denoted $A^{-1}$, is defined as the unique matrix such that $A A^{-1} = A^{-1} A = I_n$, where $I_n$ is the identity matrix of size $n \times n$.

---

A matrix is said to be **invertible** (or **non-singular**) if and only if its determinant is non-zero. That is, $A$ is invertible if $\det(A) \ne 0$.

In the case of real or complex matrices, the inverse can be computed explicitly using the **adjugate formula**:
$$
A^{-1} = \frac{1}{\det(A)} \cdot \mathrm{adj}(A),
$$
where $\mathrm{adj}(A)$ is the **adjugate matrix**, defined as the transpose of the cofactor matrix of $A$.

---

**Key properties** of the inverse include:
- The inverse of the inverse is the original matrix: $(A^{-1})^{-1} = A$.
- The inverse of a product reverses the order: $(AB)^{-1} = B^{-1} A^{-1}$.
- The inverse of the transpose is the transpose of the inverse: $(A^T)^{-1} = (A^{-1})^T$.
- The determinant of the inverse satisfies $\det(A^{-1}) = 1 / \det(A)$.

---

**Example**:  
Consider the matrix
$$
A = \begin{pmatrix}
4 & 7 \\
2 & 6
\end{pmatrix}.
$$
Its determinant is $\det(A) = 4 \cdot 6 - 7 \cdot 2 = 24 - 14 = 10$. Since the determinant is non-zero, the matrix is invertible. Applying the formula, we find:
$$
A^{-1} = \frac{1}{10}
\begin{pmatrix}
6 & -7 \\
-2 & 4
\end{pmatrix}
=
\begin{pmatrix}
0.6 & -0.7 \\
-0.2 & 0.4
\end{pmatrix}.
$$

**Assertion**: The product $A A^{-1}$ (or $A^{-1} A$) recovers the identity matrix, confirming that the inverse is correct.


In [51]:
from tensorium import Matrix, tns

print("Test 1: Identity Matrix inverse")
u1 = Matrix(3, 3)
u1.fill([
    [1., 0., 0.],
    [0., 1., 0.],
    [0., 0., 1.]
])
print(tns.inverse_mat(u1))

print("Test 2: Scalar × Identity Matrix inverse")
u2 = Matrix(3, 3)
u2.fill([
    [2., 0., 0.],
    [0., 2., 0.],
    [0., 0., 2.]
])
print(tns.inverse_mat(u2))

print("Test 3: Arbitrary 3x3 Matrix inverse")
u3 = Matrix(3, 3)
u3.fill([
    [8., 5., -2.],
    [4., 7., 20.],
    [7., 6., 1.],
])
print(tns.inverse_mat(u3))

print("Test 4: 2x2 Matrix inverse")
u4 = Matrix(2, 2)
u4.fill([
    [4.0, 7.0],
    [2.0, 6.0]
])
print(tns.inverse_mat(u4))

I = tns.mul(u1, tns.inverse_mat(u1)) 
print("Should be identity:\n", I)



Test 1: Identity Matrix inverse
[
  [1, 0, 0],
  [0, 1, 0],
  [0, 0, 1]
]
Test 2: Scalar × Identity Matrix inverse
[
  [0.5, 0, 0],
  [0, 0.5, 0],
  [0, 0, 0.5]
]
Test 3: Arbitrary 3x3 Matrix inverse
[
  [0.649425, 0.0977011, -0.655172],
  [-0.781609, -0.126437, 0.965517],
  [0.143678, 0.0747126, -0.206897]
]
Test 4: 2x2 Matrix inverse
[
  [0.6, -0.7],
  [-0.2, 0.4]
]
Should be identity:
 [
  [1, 0, 0],
  [0, 1, 0],
  [0, 0, 1]
]


#### Rank of a Matrix

The **rank** of a matrix is the dimension of the vector space spanned by its rows or its columns. More formally, if $A \in \mathbb{K}^{m \times n}$ is a matrix over a field, then the rank of $A$, denoted $\mathrm{rank}(A)$, is defined as the dimension of the image of the associated linear transformation:
$$
\mathrm{rank}(A) := \dim(\mathrm{Im}(A)).
$$
This corresponds to the maximum number of linearly independent rows or columns of $A$.

---

Equivalently, the rank of $A$ can be characterized in several ways:
- it is the dimension of the **column space** or the **row space**,
- it equals the number of non-zero rows in the row echelon form of $A$,
- it is the size of the largest square submatrix of $A$ with non-zero determinant (i.e., a non-zero minor).

---

**Important properties** include the following:
- The rank always satisfies $\mathrm{rank}(A) \leq \min(m, n)$.
- The rank is invariant under transposition: $\mathrm{rank}(A) = \mathrm{rank}(A^T)$.
- If $A$ is a square matrix of size $n$, then **$A$ is invertible if and only if $\mathrm{rank}(A) = n$**.
- For a product of matrices, we have the inequality $\mathrm{rank}(AB) \leq \min(\mathrm{rank}(A), \mathrm{rank}(B))$.

---

From a geometric point of view, the rank of $A$ represents the number of linearly independent directions in the output of the transformation $A\vec{x}$. It gives the dimension of the image of $A$, and therefore determines whether $A$ compresses space, flattens it, or spans its full output space.

---

**Example**:  
Consider the matrix
$$
A = \begin{pmatrix}
1 & 2 & 3 \\
2 & 4 & 6 \\
0 & 0 & 1
\end{pmatrix}.
$$
We observe that the second row is a linear multiple of the first, since $\text{row}_2 = 2 \cdot \text{row}_1$, which implies that it adds no new direction to the row space. On the other hand, the third row is linearly independent from the first two. Therefore, we conclude:
$$
\mathrm{rank}(A) = 2.
$$

**Assertion**: this matrix spans a two-dimensional subspace of $\mathbb{K}^3$ despite having three rows and three columns, due to the linear dependency between the first and second rows.


In [52]:
from tensorium import Matrix, tns

A = Matrix(3, 3)
A.fill([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [0.0, 0.0, 1.0]
])
print("Rank of A =", tns.rank_mat(A))

u1 = Matrix(3, 3)
u1.fill([
    [1., 0., 0.],
    [0., 1., 0.],
    [0., 0., 1.]
])
print("Rank of u1 =", tns.rank_mat(u1))  # Expected: 3

u2 = Matrix(3, 4)
u2.fill([
    [ 1., 2., 0., 0.],
    [ 2., 4., 0., 0.],
    [-1., 2., 1., 1.],
])
print("Rank of u2 =", tns.rank_mat(u2))  # Expected: 2

u3 = Matrix(4, 3)
u3.fill([
    [ 8., 5., -2.],
    [ 4., 7., 20.],
    [ 7., 6., 1.],
    [21., 18., 7.],
])
print("Rank of u3 =", tns.rank_mat(u3))  # Expected: 3



Rank of A = 2
Rank of u1 = 3
Rank of u2 = 2
Rank of u3 = 3


### Polynomial Regression Example (Quadratic Fit)

In this example, we fit a second-degree polynomial model to noisy observations of the function $y = x^2$.  
We consider input data points $x = [1, 2, 3, 4]$ and target values $y = [1.0,\ 4.1,\ 9.2,\ 16.3]$.

We aim to approximate $y$ using a model of the form:
$$
y \approx w_0 + w_1 x + w_2 x^2
$$

This is a **linear regression model in the parameters** $w = [w_0,\ w_1,\ w_2]^T$, even though it models a nonlinear function.  
To do this, we construct a design matrix $X \in \mathbb{R}^{n \times 3}$ such that:
$$
X = \begin{pmatrix}
1 & x_1 & x_1^2 \\
1 & x_2 & x_2^2 \\
\vdots & \vdots & \vdots \\
1 & x_n & x_n^2
\end{pmatrix}
$$

The optimal parameters (in the least squares sense) are obtained by solving the **normal equations**:
$$
X^T X w = X^T y
$$

We solve this system using Gaussian elimination (no explicit inversion required).  
With our data, the resulting coefficients are:

$$
w_0 \approx -0.1, \quad w_1 \approx 0.1, \quad w_2 = 1.0
$$

Thus, the final model is:
$$
\hat{y}(x) = -0.1 + 0.1x + x^2
$$

This closely matches the underlying quadratic trend, with slight corrections introduced by the noise in the data.

**Interpretation**:
- The dominant coefficient $w_2 = 1.0$ confirms that the data follows a quadratic behavior.
- The small linear term $w_1$ and offset $w_0$ correct for minor asymmetry or bias introduced by noise.


In [53]:
from tensorium import Matrix, Vector, tns, Vectord
x_data = [1.0, 2.0, 3.0, 4.0]
y_data = [1.0, 4.1, 9.2, 16.3]

n = len(x_data)
X = Matrix(n, 3)
for i in range(n):
    X[i, 0] = 1.0
    X[i, 1] = x_data[i]
    X[i, 2] = x_data[i] ** 2

y = Vector(y_data)

XT = tns.transpose_mat(X)
XT_X = tns.mul(XT, X)
XT_y = tns.mul_vec(XT, y)

w = tns.gauss_solve(XT_X, XT_y)

print("Coef :", w)

x5 = 1.0 * w[0] + 5.0 * w[1] + 25.0 * w[2]
print(f"Prediction at x = 5: {x5:.3f}")


Coef : [-0.100002, 0.100001, 1]
Prediction at x = 5: 25.400
